In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("data/committee_results.csv")


In [3]:
df_valid = df[df["agreement_score"] == "3/3"].copy()


In [4]:
df_ground_truth = df_valid.drop_duplicates(
    subset=["question", "entity1", "entity2", "relationship_type"]
).reset_index(drop=True)

In [5]:
print(f"Total rows in committee results: {len(df)}")
print(f"3/3 agreement rows (pre-dedup): {len(df_valid)}")
print(f"Unique pairs after dedup: {len(df_ground_truth)}")
df_ground_truth.head()

Total rows in committee results: 8682
3/3 agreement rows (pre-dedup): 4956
Unique pairs after dedup: 1335


,question,entity1,entity2,relationship_type,agent,score,reason,agreement_score,committee_label
0,"pulsating in my right collarbone, concerned ab...",pulmonary embolism,shortness of breath,disease-symptom,gpt-4o,1,Shortness of breath is a well-established symp...,3/3,VALID
1,"pulsating in my right collarbone, concerned ab...",pulmonary embolism,chest pain,disease-symptom,gpt-4o,1,Chest pain is a well-documented symptom of pul...,3/3,VALID
2,"pulsating in my right collarbone, concerned ab...",chiari 1,difficulty swallowing,disease-symptom,gpt-4o,1,Difficulty swallowing is a known symptom of Ch...,3/3,VALID
3,"pulsating in my right collarbone, concerned ab...",chiari 1,hoarseness,disease-symptom,gpt-4o,1,Hoarseness is a recognized symptom of Chiari t...,3/3,VALID
4,"pulsating in my right collarbone, concerned ab...",chiari 1,loss sensation,disease-symptom,gpt-4o,1,Loss of sensation can be a symptom of Chiari t...,3/3,VALID


In [6]:
def generate_questions(row):
    e1 = row["entity1"]
    e2 = row["entity2"]
    rel = row["relationship_type"]
    
    if rel == "disease-symptom":
        q1 = f"Is {e1} a disease?"
        q2 = f"Is {e2} a symptom?"
        q3 = f"Is {e2} a symptom of {e1}?"
    elif rel == "symptom-disease":
        q1 = f"Is {e1} a symptom?"
        q2 = f"Is {e2} a disease?"
        q3 = f"Is {e1} a symptom of {e2}?"
    elif rel == "drug-disease":
        q1 = f"Is {e1} a drug?"
        q2 = f"Is {e2} a disease?"
        q3 = f"Is {e1} used to treat {e2}?"
    else:
        q1 = q2 = q3 = None  # flag unexpected relationship types
    
    return pd.Series([q1, q2, q3])

df_ground_truth[["question1", "question2", "question3"]] = df_ground_truth.apply(generate_questions, axis=1)

# sanity check: any unexpected relationship types slipped through?
print(df_ground_truth["relationship_type"].unique())
df_ground_truth[["entity1", "entity2", "relationship_type", "question1", "question2", "question3"]].head()

['disease-symptom' 'symptom-disease' 'drug-disease']


,entity1,entity2,relationship_type,question1,question2,question3
0,pulmonary embolism,shortness of breath,disease-symptom,Is pulmonary embolism a disease?,Is shortness of breath a symptom?,Is shortness of breath a symptom of pulmonary ...
1,pulmonary embolism,chest pain,disease-symptom,Is pulmonary embolism a disease?,Is chest pain a symptom?,Is chest pain a symptom of pulmonary embolism?
2,chiari 1,difficulty swallowing,disease-symptom,Is chiari 1 a disease?,Is difficulty swallowing a symptom?,Is difficulty swallowing a symptom of chiari 1?
3,chiari 1,hoarseness,disease-symptom,Is chiari 1 a disease?,Is hoarseness a symptom?,Is hoarseness a symptom of chiari 1?
4,chiari 1,loss sensation,disease-symptom,Is chiari 1 a disease?,Is loss sensation a symptom?,Is loss sensation a symptom of chiari 1?


In [7]:
# Build final output with only the required columns
df_output = df_ground_truth[["entity1", "entity2", "relationship_type", "question1", "question2", "question3"]]

# Save to CSV
df_output.to_csv("data/ground_truth_qa.csv", index=False)

print(f"Saved {len(df_output)} rows to data/ground_truth_qa.csv")
df_output.head()

Saved 1335 rows to data/ground_truth_qa.csv


,entity1,entity2,relationship_type,question1,question2,question3
0,pulmonary embolism,shortness of breath,disease-symptom,Is pulmonary embolism a disease?,Is shortness of breath a symptom?,Is shortness of breath a symptom of pulmonary ...
1,pulmonary embolism,chest pain,disease-symptom,Is pulmonary embolism a disease?,Is chest pain a symptom?,Is chest pain a symptom of pulmonary embolism?
2,chiari 1,difficulty swallowing,disease-symptom,Is chiari 1 a disease?,Is difficulty swallowing a symptom?,Is difficulty swallowing a symptom of chiari 1?
3,chiari 1,hoarseness,disease-symptom,Is chiari 1 a disease?,Is hoarseness a symptom?,Is hoarseness a symptom of chiari 1?
4,chiari 1,loss sensation,disease-symptom,Is chiari 1 a disease?,Is loss sensation a symptom?,Is loss sensation a symptom of chiari 1?
